# Dataset Cleaning and Quality Analysis

In this notebook, we inspect the processed MediaPipe landmark dataset before training.

Goals:
- Verify dataset integrity
- Detect missing or corrupted samples
- Analyze class distribution
- Check landmark consistency
- Identify rare classes
- Prepare the dataset for feature engineering and model training

Dataset:
- WLASL MediaPipe Processed Dataset (MuteMotion)

## Step 1 — Load Dataset

Load the processed landmark file and metadata required for cleaning.

In [4]:
import numpy as np
import json
from pathlib import Path
from collections import Counter
DATASET = Path("../../../datasets/mutemotion")

landmarks = np.load(
    DATASET / "landmarks_V3.npz",
    allow_pickle=True
)

with open(DATASET / "WLASL_parsed_data.json", "r") as f:
    metadata = json.load(f)

print("Total Samples :", len(metadata))
print("Landmark Arrays :", len(landmarks.files))

Total Samples : 21083
Landmark Arrays : 21083


## Step 2 — Verify Sample Count

The metadata and landmark dataset should contain the same number of samples.

In [5]:
print(len(metadata) == len(landmarks.files))

True


## Step 3 — Inspect Landmark Shapes

Each sample consists of multiple frames.

Each frame contains **553 MediaPipe landmarks**:

- Face: **468**
- Pose: **33**
- Left Hand: **21**
- Right Hand: **21**

**Total:** 553 landmarks

Each landmark stores three coordinates:

- **x**
- **y**
- **z**

Since the dataset contains over **21,000 samples**, inspecting every sample is computationally expensive. Therefore, we randomly inspect a subset of the dataset to verify that the landmark structure is consistent while keeping the analysis efficient.

In [6]:
import random

SAMPLE_SIZE = 500

sample_keys = random.sample(landmarks.files, SAMPLE_SIZE)

shapes = [landmarks[key].shape for key in sample_keys]

print(f"Randomly inspected {SAMPLE_SIZE} samples.\n")

print("Unique Shapes Found:")
print(set(shapes))

Randomly inspected 500 samples.

Unique Shapes Found:
{(31, 553, 3), (128, 553, 3), (77, 553, 3), (100, 553, 3), (90, 553, 3), (40, 553, 3), (113, 553, 3), (39, 553, 3), (25, 553, 3), (52, 553, 3), (126, 553, 3), (98, 553, 3), (111, 553, 3), (185, 553, 3), (83, 553, 3), (23, 553, 3), (32, 553, 3), (129, 553, 3), (35, 553, 3), (97, 553, 3), (91, 553, 3), (81, 553, 3), (54, 553, 3), (104, 553, 3), (94, 553, 3), (53, 553, 3), (92, 553, 3), (140, 553, 3), (15, 553, 3), (112, 553, 3), (75, 553, 3), (34, 553, 3), (24, 553, 3), (51, 553, 3), (74, 553, 3), (80, 553, 3), (37, 553, 3), (87, 553, 3), (62, 553, 3), (46, 553, 3), (105, 553, 3), (45, 553, 3), (101, 553, 3), (33, 553, 3), (60, 553, 3), (17, 553, 3), (73, 553, 3), (57, 553, 3), (119, 553, 3), (103, 553, 3), (116, 553, 3), (44, 553, 3), (28, 553, 3), (43, 553, 3), (66, 553, 3), (56, 553, 3), (96, 553, 3), (59, 553, 3), (102, 553, 3), (125, 553, 3), (84, 553, 3), (115, 553, 3), (14, 553, 3), (64, 553, 3), (36, 553, 3), (26, 553, 3), (12

In [7]:
from collections import Counter

shape_counts = Counter(shapes)

print("Most Common Shapes:\n")

for shape, count in shape_counts.most_common(15):
    print(f"{shape} --> {count} samples")

Most Common Shapes:

(55, 553, 3) --> 24 samples
(53, 553, 3) --> 15 samples
(72, 553, 3) --> 14 samples
(43, 553, 3) --> 14 samples
(35, 553, 3) --> 12 samples
(64, 553, 3) --> 12 samples
(61, 553, 3) --> 11 samples
(78, 553, 3) --> 10 samples
(66, 553, 3) --> 10 samples
(80, 553, 3) --> 10 samples
(38, 553, 3) --> 9 samples
(83, 553, 3) --> 9 samples
(47, 553, 3) --> 9 samples
(62, 553, 3) --> 9 samples
(58, 553, 3) --> 9 samples


## Step 4 — Analyze Sequence Length Distribution

Each sample contains a different number of frames.

Understanding the distribution of sequence lengths helps us decide how to preprocess the data before training.

Possible preprocessing strategies include:

- Padding shorter sequences
- Truncating longer sequences
- Uniform frame sampling

This analysis helps determine an appropriate fixed sequence length for the model.

In [8]:
sequence_lengths = []

for key in random.sample(landmarks.files, 1000):
    sequence_lengths.append(landmarks[key].shape[0])

print("Samples analyzed:", len(sequence_lengths))

Samples analyzed: 1000
